# Model Evaluation, Validation & Calibration Laboratory

A model that optimizes training loss without rigorous out-of-sample diagnostic evaluation provides no guarantee of real-world performance. This lab covers threshold selection, ROC vs PR curves under severe class imbalance, probability calibration, and bias-variance diagnostics.

### Table of Contents
1. [Confusion Matrix & Decision Threshold Tuning](#1-confusion-matrix--decision-threshold-tuning)
2. [ROC-AUC vs. PR-AUC under Severe Class Imbalance](#2-roc-auc-vs-pr-auc-under-severe-class-imbalance)
3. [Probability Calibration: Reliability Diagrams & Brier Score](#3-probability-calibration-reliability-diagrams--brier-score)
4. [Learning Curves: Diagnosing High Bias vs. High Variance](#4-learning-curves-diagnosing-high-bias-vs-high-variance)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve, auc, precision_recall_curve, brier_score_loss
from sklearn.calibration import calibration_curve

sns.set_theme(style="whitegrid")
print("Loaded Model Evaluation interactive lab environment.")

## 1. Confusion Matrix & Decision Threshold Tuning
Default 0.5 decision thresholds fail when false negatives carry vastly different business costs than false positives (e.g., fraud or cancer detection).

In [ ]:
# Synthetic classification data
np.random.seed(42)
X, y = make_classification(n_samples=1000, n_features=10, weights=[0.9, 0.1], random_state=42)
clf = LogisticRegression().fit(X[:700], y[:700])
probs = clf.predict_proba(X[700:])[:, 1]
y_test = y[700:]

precisions, recalls, thresholds = precision_recall_curve(y_test, probs)

plt.figure(figsize=(7, 4.5))
plt.plot(thresholds, precisions[:-1], label="Precision", color="#1f77b4", lw=2)
plt.plot(thresholds, recalls[:-1], label="Recall", color="#d62728", lw=2, linestyle="--")
plt.title("Precision-Recall Trade-off across Decision Thresholds", fontweight="bold")
plt.xlabel("Classification Threshold")
plt.ylabel("Score")
plt.legend()
plt.tight_layout()
plt.show()

## 2. ROC-AUC vs. PR-AUC under Severe Class Imbalance
When negative instances vastly outnumber positives (e.g., 99:1), the False Positive Rate $\text{FPR} = \frac{\text{FP}}{\text{FP} + \text{TN}}$ remains deceptively tiny because of the massive $\text{TN}$ denominator, yielding an inflated ROC curve. Precision-Recall curves focus strictly on the minority class.

In [ ]:
fpr, tpr, _ = roc_curve(y_test, probs)
roc_score = auc(fpr, tpr)
pr_score = auc(recalls, precisions)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(fpr, tpr, color="#2b5c8f", lw=2, label=f"ROC Curve (AUC = {roc_score:.3f})")
axes[0].plot([0, 1], [0, 1], linestyle=":", color="gray")
axes[0].set_title("ROC Curve (Deceptively High)", fontweight="bold")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].legend()

axes[1].plot(recalls, precisions, color="#d95f02", lw=2, label=f"PR Curve (AUC = {pr_score:.3f})")
axes[1].axhline(np.mean(y_test), linestyle=":", color="gray", label="Baseline Prevalence (10%)")
axes[1].set_title("Precision-Recall Curve (True Minority Performance)", fontweight="bold")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].legend()
plt.tight_layout()
plt.show()

## 3. Probability Calibration: Reliability Diagrams & Brier Score
A well-calibrated model satisfies: among samples predicted with probability 0.8, exactly 80% should belong to the positive class.

In [ ]:
prob_true, prob_pred = calibration_curve(y_test, probs, n_bins=10)
brier = brier_score_loss(y_test, probs)

plt.figure(figsize=(7, 4.5))
plt.plot(prob_pred, prob_true, marker="o", lw=2, color="#7570b3", label=f"Logistic Regression (Brier = {brier:.4f})")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Perfect Calibration")
plt.title("Calibration Curve (Reliability Diagram)", fontweight="bold")
plt.xlabel("Mean Predicted Probability")
plt.ylabel("Fraction of True Positives")
plt.legend()
plt.tight_layout()
plt.show()